## Imports

In [1]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

## Loading dataset

In [3]:
# Loading datasets and merging them
sentences_train = pd.read_csv('datasets/valueeval24/training-english/sentences.tsv', sep='\t')
labels_train = pd.read_csv('datasets/valueeval24/training-english/labels.tsv', sep='\t')

# Merge the two files on 'text-ID' and 'sentence-ID'
train_data = pd.merge(sentences_train, labels_train, on=['Text-ID', 'Sentence-ID'])

# Repeat for the test set
sentences_test = pd.read_csv('datasets/valueeval24/test-english/sentences.tsv', sep='\t')
labels_test = pd.read_csv('datasets/valueeval24/test-english/labels.tsv', sep='\t')

test_data = pd.merge(sentences_test, labels_test, on=['Text-ID', 'Sentence-ID'])

In [4]:
# Convert train and test DataFrames into Hugging Face datasets
train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

# Remove columns that are not needed (like text-ID and sentence-ID)
train_dataset = train_dataset.remove_columns(["Text-ID", "Sentence-ID"])
test_dataset = test_dataset.remove_columns(["Text-ID", "Sentence-ID"])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Tokenizing

In [14]:
# Initialize the tokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Tokenize function for dataset
def tokenize_function(examples):
    return tokenizer(examples['Text'], padding="max_length", truncation=True, max_length=128)

# Tokenize the train and test data
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/44758 [00:00<?, ? examples/s]

Map:   0%|          | 0/14569 [00:00<?, ? examples/s]

In [15]:
# Convert labels into a tensor format
label_columns = [col for col in train_dataset.column_names if col not in ["Text", "input_ids", "attention_mask"]]
train_labels_df = train_dataset.to_pandas()[label_columns]
train_labels = torch.tensor(train_labels_df.values, dtype=torch.float32)
test_labels_df = test_dataset.to_pandas()[label_columns]
test_labels = torch.tensor(test_labels_df.values, dtype=torch.float32)

train_labels_list = train_labels.tolist()
test_labels_list = test_labels.tolist()

# Add these lists back to the dataset as the "labels" column
train_dataset = train_dataset.add_column("labels", train_labels_list)
test_dataset = test_dataset.add_column("labels", test_labels_list)

# Set the datasets to PyTorch format
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

## Loading model

In [16]:
# Load the RoBERTa model for sequence classification
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=38)

# Replace the default loss with BCEWithLogitsLoss to handle multi-label output
model.config.problem_type = "multi_label_classification"

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Training parameters

In [17]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",        # Evaluate every epoch
    save_strategy="epoch",              # Save model checkpoint every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,                 # Adjust based on needs
    weight_decay=0.01,
    load_best_model_at_end=True,        # Load best model at end of training
    metric_for_best_model="f1",         # Choose metric if you track it in evaluation
)

def compute_metrics(pred):
    # Separate logits and labels from the `pred` object
    logits, labels = pred
    
    # Apply sigmoid to logits to obtain probabilities
    logits = torch.sigmoid(torch.tensor(logits)).cpu().numpy()
    
    # Convert probabilities to binary predictions (0 or 1) based on a threshold of 0.5
    predictions = (logits > 0.5).astype(int)

    print("Predictions:", predictions)
    print("Labels:", labels)
    print("Predictions Shape:", predictions.shape)
    print("Labels Shape:", labels.shape)
    
    # Labels are already in numpy format, so we don't need .detach() here
    # Calculate metrics
    f1 = f1_score(labels, predictions, average="micro")
    accuracy = accuracy_score(labels, predictions)
    
    return {"f1": f1, "accuracy": accuracy}

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Training

In [18]:
# Set up Trainer for model fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Fine-tune the model
trainer.train()

  0%|          | 0/33570 [00:00<?, ?it/s]

{'loss': 0.1311, 'grad_norm': 0.2697562575340271, 'learning_rate': 1.9702114983616326e-05, 'epoch': 0.04}
{'loss': 0.0742, 'grad_norm': 0.24694442749023438, 'learning_rate': 1.940422996723265e-05, 'epoch': 0.09}
{'loss': 0.0737, 'grad_norm': 0.24678967893123627, 'learning_rate': 1.9106344950848975e-05, 'epoch': 0.13}
{'loss': 0.0709, 'grad_norm': 0.3872770667076111, 'learning_rate': 1.8808459934465297e-05, 'epoch': 0.18}
{'loss': 0.0718, 'grad_norm': 0.3120473623275757, 'learning_rate': 1.8510574918081624e-05, 'epoch': 0.22}
{'loss': 0.0721, 'grad_norm': 0.3173730969429016, 'learning_rate': 1.8212689901697946e-05, 'epoch': 0.27}
{'loss': 0.072, 'grad_norm': 0.2742948830127716, 'learning_rate': 1.791480488531427e-05, 'epoch': 0.31}
{'loss': 0.0689, 'grad_norm': 0.2764975130558014, 'learning_rate': 1.7616919868930595e-05, 'epoch': 0.36}
{'loss': 0.0663, 'grad_norm': 0.10977157205343246, 'learning_rate': 1.731903485254692e-05, 'epoch': 0.4}
{'loss': 0.0665, 'grad_norm': 0.6856733560562134

  0%|          | 0/3643 [00:00<?, ?it/s]

Predictions: [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
Labels: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Predictions Shape: (14569, 38)
Labels Shape: (14569, 38)


ValueError: Classification metrics can't handle a mix of continuous-multioutput and multilabel-indicator targets

In [ ]:
# Evaluate on the test set
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

## Testing

In [ ]:
def predict(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities > 0.5).int()
    return predictions

# Example prediction
text = "Your input sentence here"
prediction = predict(model, tokenizer, text)
print("Predicted labels:", prediction)